In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
MODELSDIR = CONFIGS['filepaths']['models']
SRMODELS  = CONFIGS['experiments']['sr']['optimizedeqs']
SPLIT     = 'test'

REGISTRY = {row['name']:dict(form=row['form'],constants=json.loads(row['constants']),
                              train_loss=row['train_loss'],valid_loss=row['valid_loss'])
             for _,row in pd.read_csv(os.path.join(MODELSDIR,'sr','optimized_equations.csv')).iterrows()}

with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
SIGMA = STATS['tp_std']
MEAN  = STATS['tp_mean']
ZMIN  = -MEAN / SIGMA

## SR-BL: physical-space conversion

The normalized SR-BL equation discovered by symbolic regression is:

$$
f(\hat{B}_L) = (\hat{B}_L + a)^3 + b, \qquad \hat{B}_L = \frac{B_L - \mu_B}{\sigma_B}.
$$

Substituting the standardization and collecting constants yields the **physical-space** form:

$$
\boxed{P(B_L) = \max\!\Big(\exp\!\big(\alpha\,(B_L - B_L^*)^3 + \beta\big) - 1,\; 0\Big),}
$$

with three interpretable constants:

| Constant | Formula | Meaning |
|:--------:|:--------|:--------|
| $B_L^*$ | $\mu_B - a\,\sigma_B$ | Critical buoyancy threshold |
| $\alpha$ | $s_y / \sigma_B^3$ | Exponential sensitivity to departures from $B_L^*$ |
| $\beta$ | $s_y \cdot b$ | Log-space offset controlling overall magnitude |

Precipitation onset ($P > 0$) occurs when $B_L > B_L^* - b^{1/3}\,\sigma_B$.

In [ ]:
C = REGISTRY['sr_bl_eq']['constants']
a,b = C['a'],C['b']
mu_B  = STATS['bl_mean']
sig_B = STATS['bl_std']

BLSTAR = mu_B - a * sig_B
ALPHA  = SIGMA / sig_B**3
BETA   = SIGMA * b
ONSET  = BLSTAR - b**(1/3) * sig_B

print(f'Normalized constants:  a = {a},  b = {b}')
print(f'Training statistics:   mu_B = {mu_B:.6f} m/s²,  sig_B = {sig_B:.6f} m/s²')
print(f'                      s_y  = {SIGMA:.6f}')
print()
print(f'Physical constants:')
print(f'  B_L*  = {BLSTAR:.4f} m/s²')
print(f'  alpha = {ALPHA:.2f} (m/s²)^-3')
print(f'  beta  = {BETA:.4f}')
print(f'  Onset = {ONSET:.4f} m/s²')

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    blphys = ds['bl'].transpose('time','lat','lon').values.ravel()
    obs    = ds['tp'].transpose('time','lat','lon').values.ravel()

valid = np.isfinite(blphys) & np.isfinite(obs)
blphys = blphys[valid]
obs    = obs[valid]

print(f'Loaded {valid.sum():,} valid samples')
print(f'B_L range: [{blphys.min():.4f}, {blphys.max():.4f}] m/s²')
print(f'Observed precip range: [{obs.min():.2f}, {obs.max():.2f}] mm')

In [ ]:
def predict_srbl_physical(bl):
    return np.maximum(np.expm1(ALPHA * (bl - BLSTAR)**3 + BETA),0.0)

def bin_1d(x,z,nbins=30,minsamples=50,plo=1,phi=99):
    mask = np.isfinite(x) & np.isfinite(z)
    x,z  = x[mask],z[mask]
    edges  = np.linspace(*np.percentile(x,[plo,phi]),nbins+1)
    xi     = np.clip(np.digitize(x,edges)-1,0,nbins-1)
    counts = np.bincount(xi,minlength=nbins)
    sums   = np.bincount(xi,weights=z,minlength=nbins)
    return 0.5*(edges[:-1]+edges[1:]),np.where(counts>=minsamples,sums/counts,np.nan),counts

blrange = np.linspace(np.percentile(blphys,1),np.percentile(blphys,99),300)
pred    = predict_srbl_physical(blrange)
centers,obsmean,_ = bin_1d(blphys,obs,nbins=30)

In [ ]:
COLOR = SRMODELS['sr_bl_eq']['color']

fig,ax = pplt.subplots(figwidth=4.0,refheight=2.8)

ax.scatter(centers,obsmean,color='k',s=14,zorder=3,label='ERA5')
ax.plot(blrange,pred,color=COLOR,linewidth=2.2,label='SR-BL',zorder=2)

ax.axvline(BLSTAR,color='gray',linewidth=1.0,linestyle='--',alpha=0.7,zorder=1)
ax.text(BLSTAR,ax.get_ylim()[1]*0.92,r'$B_L^*$',ha='right',va='top',
        fontsize=10,color='gray',fontweight='bold',
        transform=ax.get_xaxis_transform())

ax.axvline(ONSET,color='gray',linewidth=0.8,linestyle=':',alpha=0.5,zorder=1)
ax.text(ONSET,ax.get_ylim()[1]*0.78,r'onset',ha='right',va='top',
        fontsize=8,color='gray',fontstyle='italic',
        transform=ax.get_xaxis_transform())

ax.annotate(r'$P = \max\left(e^{\alpha(B_L - B_L^*)^3 + \beta} - 1,\; 0\right)$',
            xy=(0.97,0.55),xycoords='axes fraction',ha='right',va='top',
            fontsize=8.5,color=COLOR,
            bbox=dict(boxstyle='round,pad=0.3',fc='white',ec='none',alpha=0.8))

ax.format(grid=False,
          xlabel=r'Buoyancy Measure $B_L$ (m s$^{-2}$)',
          ylabel='Precipitation (mm)')
ax.legend(loc='ul',ncols=1)
fig.save('../figs/fig_srbl_physical.jpg')